# Simulateur de sessions de recharge

Conception et test de la logique de génération de sessions simulées :
tirage de l'heure de début (pondérée vers la soirée), choix du
connecteur, calcul de l'énergie et de la durée (ajustée par la
température), génération d'un batch complet.

La logique a depuis été migrée vers `src/simulation/sessions.py`, avec
tests unitaires associés (`tests/test_simulation_sessions.py`).

Ce notebook sert maintenant d'exemple d'usage du module, incluant la
génération et le chargement d'un batch de 455 sessions dans DuckDB.

In [1]:
import logging
from datetime import date
from pathlib import Path

import duckdb
import polars as pl

from simulation.sessions import generer_session
from warehouse.duckdb_loader import charger_sessions_dans_duckdb

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")

ROOT_PATH = Path.cwd().resolve().parent
con = duckdb.connect(str(ROOT_PATH / "data" / "warehouse" / "electric_mobility.duckdb"))

connections_df = con.execute("SELECT * FROM connections").pl()
connections_operationnelles = connections_df.filter(pl.col("is_operational"))

NB_SESSIONS = 455  # 99 connecteurs × ln(99) ≈ 455, problème du collectionneur de coupons

sessions_generees = [
    generer_session(con, connections_operationnelles, date(2026, 7, 20), date(2026, 7, 24))
    for _ in range(NB_SESSIONS)
]
sessions_df = pl.DataFrame(sessions_generees)

charger_sessions_dans_duckdb(con, sessions_df)

2026-08-13 11:11:17,895 - warehouse.duckdb_loader - INFO - Table sessions créée avec succès.
2026-08-13 11:11:18,028 - warehouse.duckdb_loader - INFO - Insertion données dans la table sessions : 455 lignes traitées avec succès
2026-08-13 11:11:18,030 - warehouse.duckdb_loader - INFO - Chargement DuckDB terminé : sessions=(455, 4)


In [2]:
con.close()